In [2]:
# selenium, Beautifulsoup 임포트
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
import time

# 대기 시간 변수 선언
sleep_time = 2

In [4]:
# 데이터별 리스트 초기화(시/도 구분없이 한 리스트에 전부 저장)
name_lst = [] # 이름 리스트
address_lst = [] # 도로명 주소 리스트
g_price_lst = [] # 휘발유 가격 리스트
d_price_lst = [] # 경유 가격 리스트
phone_lst = [] # 전화번호 리스트

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/searRgOsSelect.do')
WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))

# 시/도 select_box의 전체 옵션 수
option_cnt = len(Select(driver.find_element(By.ID, 'SIDO_NM0')).options)

# 전국 시/도 착한주유소 이름, 휘발유 가격, 경유 가격, 주소 크롤링 루프문
for i in range(1, option_cnt):
    # 시/도 선택하는 select_box 객체 생성
    select_element = driver.find_element(By.ID, 'SIDO_NM0')
    select_box = Select(select_element)

    # 시/도 선택
    select_box.select_by_index(i)

    # 조회 버튼 클릭
    driver.find_element(By.ID, 'searRgSelect').click()
    time.sleep(sleep_time) # 페이지 요소들 로딩될때까지 대기
    
    # 시의 착한 주유소 세부정보 크롤링
    # 주유소 세부정보 링크 객체 리스트
    WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.rlist > a')))
    href_elements = driver.find_elements(By.CSS_SELECTOR, '.rlist > a')

    # 각 주유소 세부정보 링크 클릭 -> 세부정보 크롤링 루프문
    for i in range(len(href_elements)):
        try:
            # 링크 클릭
            href_elements[i].click()

            # Beautiful soup 객체 생성
            results_html = driver.page_source
            soup = BeautifulSoup(results_html, 'html.parser')

            # 주소 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'rd_addr')))
            address = soup.select_one('#rd_addr').text
            address_lst.append(address)

            # 이름 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'os_nm')))
            name = soup.select_one('#os_nm').text
            name_lst.append(name)

            # 휘발유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'b027_p')))
            g_price = soup.select_one('#b027_p').text
            g_price_lst.append(g_price)
            
            # 경유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'd047_p')))
            d_price = soup.select_one('#d047_p').text
            d_price_lst.append(d_price)

            # 전화번호 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'phn_no')))
            phone = soup.select_one('#phn_no').text
            phone_lst.append(phone)
        
        except:
            continue
        
# print(len(name_lst))
# print(len(g_price_lst))
# print(len(d_price_lst))
# print(len(address_lst))
# print(len(phone_lst))
# print(name_lst)
# print(g_price_lst)
# print(d_price_lst)
# print(address_lst)
# print(phone_lst)

# 시/도별로 저장하고 싶으면 루프마다 name_lst, g_price_lst, d_price_lst, phone_lst 초기화 후 db에 저장




In [16]:
# 데이터별 리스트 초기화
question_lst = []
answer_lst = []

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/user/cufaq/cufaqSelectPrice.do')
WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.t_left.input > a')))

# 전체 질문 수
question_cnt = len(driver.find_elements(By.CSS_SELECTOR, '.t_left.input > a'))
print(question_cnt)

for i in range(question_cnt):
    # FAQ 하나를 클릭
    question_element = driver.find_elements(By.CSS_SELECTOR, '.t_left.input > a')
    question_element[i].click()
    WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CLASS_NAME, 'view_contents')))

    # Beautiful soup 객체 생성
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # FAQ 질문과 답변을 각각 리스트에 저장
    question_lst.append(soup.select_one('.title_txt').text)
    answer_lst.append(soup.select_one('.view_contents').text.replace('\n', '').replace('\t', ''))

    # FAQ 목록 페이지로 돌아가기
    driver.back()
    time.sleep(sleep_time)
    

    

print(len(question_lst))
print(len(answer_lst))
print(question_lst)
print(answer_lst)



8
8
8
['오피넷에 게시되는 유가는 세금을 포함하고 있나요?', '오피넷에 나오는 가격은 어떻게 수집되나요?', '오피넷에서 유가는 어떻게 볼 수 있나요?', '오피넷 가격이 틀려요', '휘발유, 등유, 경유, LPG를 제외한 타 유종의 가격을 알고 싶어요.', '유가정보 내려받기 과거 판매가격 기준 안내', '공장도 가격과 오피넷에 공개되고 있는 정유사 또는 대리점 가격의 차이', '정유사 공급가격과 판매가격의 차이가 무엇인가요?']
["오피넷에 게시되는 모든 유가는 부가가치세 등 모든 세금이 포함된 최종 가격입니다. (세금관련 사항은 '유가관련정보-유류세' 에서 찾아보실 수 있습니다.)'주유소충전소찾기' 메뉴에서 보시는 가격은 물론,유가통계정보 메뉴에서 보시는 가격도 모두 세금을 포함하는 가격이며,정유사가격의 경우 개별 세금내역과 함께 세전, 세후 가격을 모두 보실 수 있습니다.", '오피넷은 전국 약 12,000여개 이상의 주유소와 LPG충전소를 전수조사하여 판매가격정보를 수집합니다.1. VAN사업자를 통해 신용카드 결제 단말기에서 제품별 판매 단가를 자동수집하여 제공2. 주유소 등 석유 판매업자가 직접 입력 보고(홈페이지/앱, ARS)이 중 VAN(카드 단말기)을 이용한 방법이 가장 많이 사용되고 있으며, 카드를 이용하여 결제된 정보를 하루 6회 받아오고 있습니다.00:00~01:00 => 02:00 업데이트 06:00~08:00 => 09:00 업데이트08:00~11:00 => 12:00 업데이트11:00~15:00 => 16:00 업데이트15:00~18:00 => 19:00 업데이트 18:00~24:00 => 01:00 업데이트', '개별 주유소/충전소의 판매가격을 알고 싶다면 국내유가통계>주유소/충전소>유가내려받기 메뉴, 전체 통계를 알고 싶다면 국내유가통계>주유소/충전소>평균판매가격 메뉴에서 확인하실 수 있습니다.', ' 오피넷의 가격은 주유소 사업자의 보고, 카드단말기 사용 등을 통해 수집됩니다.  따라서 주유소 사업자가 가격변동 후에도 변